## 1. Importation & Nettoyage des Données

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams.update({'figure.dpi': 130, 'font.size': 10})

# Chargement
df = pd.read_csv('global_power_plant_database.csv')
print(f"Dimensions brutes : {df.shape}")
print(f"Colonnes : {df.columns.tolist()}")


In [ ]:

# Valeurs manquantes
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(2)
pd.DataFrame({'Manquants': missing, '%': missing_pct}).sort_values('%', ascending=False).head(12)


In [ ]:

# Conversion NumPy des colonnes numériques
for col in ['capacity_mw', 'latitude', 'longitude', 'commissioning_year']:
    df[col] = pd.to_numeric(df[col], errors='coerce')

# Compléter la production réelle manquante avec les estimations
for y in [2013, 2014, 2015, 2016, 2017]:
    col_a = f'generation_gwh_{y}'; col_e = f'estimated_generation_gwh_{y}'
    if col_a in df.columns and col_e in df.columns:
        df[col_a].fillna(df[col_e], inplace=True)

# Nettoyage des lignes sans données essentielles
df.dropna(subset=['primary_fuel', 'capacity_mw'], inplace=True)
print(f"Dimensions après nettoyage : {df.shape}")
print(f"Types de combustible uniques ({df['primary_fuel'].nunique()}) : {sorted(df['primary_fuel'].unique())}")


## 2. Analyse Exploratoire des Données (EDA)

In [ ]:
# Statistiques descriptives
gen_cols = [f'generation_gwh_{y}' for y in [2013,2014,2015,2016,2017]]
df[['capacity_mw'] + gen_cols].describe().round(2)


In [ ]:

top_countries = df.groupby('country_long')['capacity_mw'].sum().sort_values(ascending=False).head(15)
top_fuels = df['primary_fuel'].value_counts().head(12)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_countries.plot(kind='barh', ax=axes[0], color=sns.color_palette("Blues_r", 15))
axes[0].set_title("Top 15 pays – Capacité totale installée (MW)", fontweight='bold')
axes[0].set_xlabel("Capacité (MW)"); axes[0].invert_yaxis()

# Combustibles les plus utilisés
top_fuels.plot(kind='bar', ax=axes[1], color=sns.color_palette("Set2", 12))
axes[1].set_title("Nb de centrales par type de combustible", fontweight='bold')
axes[1].set_xlabel("Combustible"); axes[1].set_ylabel("Nb centrales")
axes[1].tick_params(axis='x', rotation=40)
plt.tight_layout(); plt.show()
# Observation : USA, Chine, Inde dominent. Le solaire est le fuel le plus représenté en nombre.


## 3. Analyse Statistique – Puissance par Type de Carburant

In [ ]:
# Analyse par type de combustible
fuels_main = df['primary_fuel'].value_counts().head(8).index.tolist()
df_main = df[df['primary_fuel'].isin(fuels_main)]

fuel_stats = df_main.groupby('primary_fuel')['capacity_mw'].agg(
    Moyenne='mean', Médiane='median', Écart_type='std', Nb_centrales='count').round(2)
print(fuel_stats.sort_values('Moyenne', ascending=False).to_string())


In [ ]:

# ANOVA – test si les moyennes diffèrent entre carburants
groups = [df_main[df_main['primary_fuel']==f]['capacity_mw'].dropna().values for f in fuels_main]
f_stat, p_val = stats.f_oneway(*groups)
print(f"ANOVA : F = {f_stat:.2f},  p-value = {p_val:.2e}")
print("✅ Différences SIGNIFICATIVES (p < 0.05) entre types de carburant" if p_val < 0.05 else "❌ Pas de différence significative")

# Tests t pairwise (Welch) qui déterminent quelles paires de carburants diffèrent significativement
from itertools import combinations
results = []
for f1, f2 in combinations(fuels_main, 2):
    g1 = df_main[df_main['primary_fuel']==f1]['capacity_mw'].dropna()
    g2 = df_main[df_main['primary_fuel']==f2]['capacity_mw'].dropna()
    t, p = stats.ttest_ind(g1, g2, equal_var=False)
    results.append({'Paire': f'{f1} vs {f2}', 't': round(t,2), 'p': round(p,4)})
pd.DataFrame(results).sort_values('p').head(8)


In [ ]:
# Visualisation des distributions de capacité par type de combustible
fig, ax = plt.subplots(figsize=(14, 5))
order = df_main.groupby('primary_fuel')['capacity_mw'].median().sort_values(ascending=False).index
sns.boxplot(data=df_main, x='primary_fuel', y='capacity_mw', order=order,
            palette='Set3', showfliers=False, ax=ax)
ax.set_title("Distribution de la capacité (MW) par type de combustible\n(valeurs aberrantes masquées)", fontweight='bold')
ax.set_xlabel("Type de combustible"); ax.set_ylabel("Capacité (MW)")
ax.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()


## 4. Analyse des Séries Chronologiques

In [ ]:

df_year = df.dropna(subset=['commissioning_year'])
df_year = df_year[(df_year['commissioning_year'] >= 1900) & (df_year['commissioning_year'] <= 2020)].copy()
df_year['commissioning_year'] = df_year['commissioning_year'].astype(int)

# Tendance de mise en service des centrales dans le temps
yearly = df_year.groupby('commissioning_year').size().reset_index(name='count')
x, y = yearly['commissioning_year'].values, yearly['count'].values
coef = np.polyfit(x, y, 1)          # NumPy : régression linéaire
trend = np.poly1d(coef)
print(f"Tendance : +{coef[0]:.2f} nouvelles centrales par an en moyenne")


fig, ax = plt.subplots(figsize=(13, 5))
ax.bar(x, y, color='steelblue', alpha=0.55, label='Centrales/an')
ax.plot(x, trend(x), 'r--', lw=2.5, label=f'Tendance linéaire (+{coef[0]:.1f}/an)')
ax.set_title("Nombre de centrales mises en service par année", fontweight='bold')
ax.set_xlabel("Année"); ax.set_ylabel("Nombre de centrales")
ax.set_xlim(1900, 2020); ax.legend(); plt.tight_layout(); plt.show()


In [ ]:

df_year['decade'] = (df_year['commissioning_year'] // 10) * 10
top6_fuel = df_year['primary_fuel'].value_counts().head(6).index
fuel_decade = df_year.groupby(['decade','primary_fuel']).size().unstack(fill_value=0)
fuel_decade_pct = fuel_decade[top6_fuel].div(fuel_decade[top6_fuel].sum(axis=1), axis=0) * 100

# Visualisation de l'évolution de la composition énergétique par décennie
fuel_decade_pct.plot(kind='area', figsize=(13,5), colormap='tab10', alpha=0.72)
plt.title("Évolution de la composition énergétique par décennie (%)", fontweight='bold')
plt.xlabel("Décennie"); plt.ylabel("Part relative (%)")
plt.legend(loc='upper left', fontsize=9); plt.tight_layout(); plt.show()
# Observation : montée en puissance spectaculaire du solaire depuis 2010.


## 5. Visualisation Avancée

In [ ]:

# Carte mondiale de répartition géographique
df_geo = df[df['primary_fuel'].isin(fuels_main)].dropna(subset=['latitude','longitude'])
fuel_colors = {f: c for f, c in zip(fuels_main, sns.color_palette("tab10", len(fuels_main)))}

fig, ax = plt.subplots(figsize=(18, 9))
for fuel in fuels_main:
    sub = df_geo[df_geo['primary_fuel'] == fuel]
    ax.scatter(sub['longitude'], sub['latitude'],
               s=np.clip(sub['capacity_mw'] / 200, 1, 40),  # NumPy clip
               c=[fuel_colors[fuel]], alpha=0.35, label=fuel, linewidths=0)
ax.set_title("Carte mondiale des centrales électriques\n(taille des points ∝ capacité MW)", fontweight='bold', fontsize=13)
ax.set_xlabel("Longitude"); ax.set_ylabel("Latitude")
ax.set_xlim(-180,180); ax.set_ylim(-90,90)
ax.axhline(0, color='gray', lw=0.5, ls='--', alpha=0.5)
ax.legend(loc='lower left', fontsize=8, markerscale=2)
ax.set_facecolor('#d6eaf8')
plt.tight_layout(); plt.show()


In [ ]:

# Heatmap de corrélation
gen_cols = [f'generation_gwh_{y}' for y in [2013,2014,2015,2016,2017]]
corr_df = df[['capacity_mw'] + gen_cols].dropna()
fig, ax = plt.subplots(figsize=(8,6))
sns.heatmap(corr_df.corr(), annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.5, ax=ax)
ax.set_title("Corrélations : Capacité & Production annuelle (GWh)", fontweight='bold')
plt.tight_layout(); plt.show()


## 6. Opérations Matricielles – ACP (Analyse en Composantes Principales)

In [ ]:

# Matrice des caractéristiques moyennes par type de combustible
fuel_numeric = df_main.groupby('primary_fuel')[['capacity_mw','latitude','longitude']].mean().dropna()
X = fuel_numeric.values
feature_names = ['capacity_mw', 'latitude', 'longitude']

# 1. Standardisation (NumPy)
X_std = (X - X.mean(axis=0)) / X.std(axis=0)

# 2. Matrice de covariance
cov = np.cov(X_std.T)
print("Matrice de covariance :\n", np.round(cov, 3))

# 3. Décomposition propre
eigenvalues, eigenvectors = np.linalg.eig(cov)
idx = np.argsort(eigenvalues)[::-1]
eigenvalues  = eigenvalues[idx].real
eigenvectors = eigenvectors[:, idx].real
explained = eigenvalues / eigenvalues.sum() * 100

print(f"\nValeurs propres : {np.round(eigenvalues, 3)}")
print(f"Variance expliquée (%) : {np.round(explained, 2)}")
for i, (val, vec) in enumerate(zip(eigenvalues, eigenvectors.T)):
    contrib = {feature_names[j]: round(vec[j],3) for j in range(len(feature_names))}
    print(f"PC{i+1} ({explained[i]:.1f}%) – contributions : {contrib}")


In [ ]:

# Projection ACP manuelle (multiplication matricielle NumPy)
X_pca = X_std @ eigenvectors[:, :2]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = sns.color_palette("tab10", len(fuel_numeric))
for i, fuel in enumerate(fuel_numeric.index):
    axes[0].scatter(X_pca[i,0], X_pca[i,1], s=140, color=colors[i], zorder=3, label=fuel)
    axes[0].annotate(fuel, (X_pca[i,0]+0.06, X_pca[i,1]), fontsize=8)
axes[0].axhline(0, color='gray', ls='--', lw=0.5)
axes[0].axvline(0, color='gray', ls='--', lw=0.5)
axes[0].set_title("ACP – Projection des types de combustible\n(capacité, latitude, longitude)", fontweight='bold')
axes[0].set_xlabel(f"PC1 ({explained[0]:.1f}% variance)")
axes[0].set_ylabel(f"PC2 ({explained[1]:.1f}% variance)")

# Variance expliquée par composante principale
axes[1].bar([f'PC{i+1}' for i in range(len(eigenvalues))], explained,
            color=sns.color_palette("pastel"))
axes[1].set_title("Variance expliquée par composante principale", fontweight='bold')
axes[1].set_ylabel("%")
plt.tight_layout(); plt.show()
# → PC1 (63%) : axe capacité + géographie.  PC2 (32%) : latitude seule.


## 7. Intégration NumPy + Pandas + Matplotlib

In [ ]:

# 7a – Filtrage complexe avec z-scores NumPy dans Pandas
mean_cap = df_main.groupby('primary_fuel')['capacity_mw'].transform('mean')
std_cap  = df_main.groupby('primary_fuel')['capacity_mw'].transform('std')
z_scores = (df_main['capacity_mw'] - mean_cap) / std_cap.replace(0, np.nan)

giants = df_main[z_scores > 2].copy()
print(f"Centrales géantes (z-score > 2 au sein de leur catégorie) : {len(giants)}")
print("\nTop 10 par capacité :")
giants[['name','country_long','primary_fuel','capacity_mw']].sort_values('capacity_mw', ascending=False).head(10)


In [ ]:

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Graphe violon avec percentiles NumPy ---
ax = axes[0]
order2 = df_main.groupby('primary_fuel')['capacity_mw'].median().sort_values(ascending=False).head(6).index.tolist()
data_v = [df_main[df_main['primary_fuel']==f]['capacity_mw'].dropna().values for f in order2]
vp = ax.violinplot(data_v, positions=range(len(order2)), showmedians=True, showextrema=False)
for i, data in enumerate(data_v):
    p25, p75 = np.percentile(data, [25, 75])   # ← NumPy percentile
    ax.plot([i-0.15, i+0.15], [p25, p25], 'k--', lw=1.2, alpha=0.7)
    ax.plot([i-0.15, i+0.15], [p75, p75], 'k--', lw=1.2, alpha=0.7)
for body in vp['bodies']: body.set_alpha(0.6)
ax.set_xticks(range(len(order2))); ax.set_xticklabels(order2, rotation=30)
ax.set_title("Violon + P25/P75 (NumPy) – Capacité MW", fontweight='bold')
ax.set_ylabel("Capacité (MW)")

# --- Courbe de Pareto avec cumsum NumPy ---
ax2 = axes[1]
top6_cap = df_main.groupby('primary_fuel')['capacity_mw'].sum().sort_values(ascending=False).head(6)
cumsum   = np.cumsum(top6_cap.values)   # ← NumPy cumsum
total    = cumsum[-1]
bars = ax2.bar(top6_cap.index, top6_cap.values, color=sns.color_palette("tab10",6), alpha=0.85)
ax2_r = ax2.twinx()
ax2_r.plot(range(len(top6_cap)), cumsum/total*100, 'ko-', lw=2, ms=7)
ax2_r.set_ylabel("% cumulatif (Pareto)")
ax2_r.set_ylim(0, 110)
for i, v in enumerate(cumsum/total*100):
    ax2_r.annotate(f"{v:.0f}%", (i, v+2), ha='center', fontsize=8)
ax2.set_title("Capacité totale & Courbe de Pareto (NumPy cumsum)", fontweight='bold')
ax2.set_ylabel("Capacité totale (MW)")
ax2.tick_params(axis='x', rotation=30)
plt.tight_layout(); plt.show()
# → Charbon + Gaz + Hydro représentent ~76% de la capacité mondiale installée.
